# Lending & Credit

- [Credit Risk Basics](#Credit-Risk-Basics)
- [Scorecards: Weight of Evidence and Score Scaling](#Scorecards:-Weight-of-Evidence-and-Score-Scaling)
  - [Scorecard validation: Gini, KS and PSI](#Scorecard-validation:-Gini,-KS-and-PSI)
- [Reject Inference: the Missing Outcomes Problem](#Reject-Inference:-the-Missing-Outcomes-Problem)
- [Fairness and Adverse Action](#Fairness-and-Adverse-Action)
- [Blogs](#Blogs)
  - [Pitfalls](#Pitfalls)

### Credit Risk Basics

A lender cares about the expected loss on a loan or a portfolio, which splits into three parts:

- **PD**, probability of default over the horizon (usually 12 months).
- **LGD**, loss given default: the share of the exposure not recovered.
- **EAD**, exposure at default: the amount owed when default happens.

Expected loss = PD x LGD x EAD. A 3% PD, 45% LGD and $10,000 exposure give an expected loss of $135 per loan. Pricing, provisioning and capital all build on this, and most lending ML work is about estimating PD.

In [1]:
import numpy as np, pandas as pd

print("one loan: expected loss = $", 0.03 * 0.45 * 10_000)
book = pd.DataFrame({"segment": ["prime", "near-prime", "subprime"],
                     "pd": [0.01, 0.03, 0.08], "lgd": [0.40, 0.45, 0.60],
                     "ead": [20_000, 10_000, 5_000], "loans": [500, 300, 100]})
book["el_per_loan"] = book.pd * book.lgd * book.ead
book["el_total"] = book.el_per_loan * book.loans
print(book)
print("portfolio expected loss: $", round(book.el_total.sum()))

one loan: expected loss = $ 135.0
      segment    pd   lgd    ead  loans  el_per_loan  el_total
0       prime  0.01  0.40  20000    500         80.0   40000.0
1  near-prime  0.03  0.45  10000    300        135.0   40500.0
2    subprime  0.08  0.60   5000    100        240.0   24000.0
portfolio expected loss: $ 104500


### Scorecards: Weight of Evidence and Score Scaling

Scorecards stay popular in lending because they are transparent: each bin of each variable adds or subtracts points. The building blocks:

- **Binning.** Cut each variable into bins, which handles outliers and missing values and captures non-linear effects.
- **Weight of Evidence (WoE).** For each bin, WoE = ln(share of all goods in the bin / share of all bads in the bin). Positive WoE means the bin is safer than average.
- **Information Value (IV).** IV = sum over bins of (share of goods - share of bads) x WoE. It ranks variables by separating power. An IV above about 0.5 is usually a leakage warning, not a great feature.
- **Logistic regression on WoE.** WoE makes each variable roughly linear in the log-odds, so a plain logistic model gives the points.
- **Scaling.** Points = offset + factor x ln(odds of good). With "points to double the odds" (PDO) of 20, factor = PDO / ln 2.

Synthetic applicants where default risk rises with credit utilization:

In [2]:
rng = np.random.default_rng(0)
n = 30000
util = rng.beta(2, 3, n)
default = rng.binomial(1, 1 / (1 + np.exp(-(-3.2 + 3.0 * util))))
df = pd.DataFrame({"util": util, "bad": default})
df["bin"] = pd.cut(df.util, [0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True)

g = df.groupby("bin", observed=True).bad.agg(bads="sum", total="size")
g["goods"] = g.total - g.bads
g["pct_good"] = g.goods / g.goods.sum()
g["pct_bad"] = g.bads / g.bads.sum()
g["woe"] = np.log(g.pct_good / g.pct_bad)
g["iv_part"] = (g.pct_good - g.pct_bad) * g.woe
print(g[["total", "bads", "pct_good", "pct_bad", "woe", "iv_part"]].round(3))
print("IV of utilization:", round(g.iv_part.sum(), 3))

               total  bads  pct_good  pct_bad    woe  iv_part
bin                                                          
(-0.001, 0.2]   5264   288     0.191    0.073  0.969    0.115
(0.2, 0.4]     10303   956     0.359    0.241  0.400    0.047
(0.4, 0.6]      9025  1354     0.295    0.341 -0.146    0.007
(0.6, 0.8]      4585  1096     0.134    0.276 -0.722    0.103
(0.8, 1.0]       823   277     0.021    0.070 -1.202    0.059
IV of utilization: 0.33


In [3]:
from sklearn.linear_model import LogisticRegression

woe_map = g.woe.to_dict()
X = df.bin.map(woe_map).astype(float).to_frame("woe_util")
lr = LogisticRegression().fit(X, 1 - df.bad)                  # model the odds of GOOD
PDO, base_score, base_odds = 20, 600, 50                      # 600 points at 50:1 good:bad odds
factor = PDO / np.log(2)
offset = base_score - factor * np.log(base_odds)
for b, w in woe_map.items():
    log_odds = lr.intercept_[0] + lr.coef_[0][0] * w
    print(f"utilization {str(b):12s} | score {offset + factor * log_odds:6.1f} | estimated bad rate {1 / (1 + np.exp(log_odds)):.3f}")
print("check: points to double the odds =", round(factor * np.log(2), 1))

utilization (-0.001, 0.2] | score  569.3 | estimated bad rate 0.055
utilization (0.2, 0.4]   | score  552.9 | estimated bad rate 0.093
utilization (0.4, 0.6]   | score  537.2 | estimated bad rate 0.150
utilization (0.6, 0.8]   | score  520.5 | estimated bad rate 0.239
utilization (0.8, 1.0]   | score  506.7 | estimated bad rate 0.336
check: points to double the odds = 20.0


Higher utilization bins score lower, and the score maps back to a bad rate. In a real scorecard you stack several variables, each contributing its own points, and keep the table so that every decision is explainable by adding up bins.

#### Scorecard validation: Gini, KS and PSI

Gini=2·AUC−1 (rank-ordering power). KS=max gap between cumulative good/bad score distributions. PSI: stability of development vs current population. Also calibration by score band, out-of-time backtesting. Typical: Gini 40–70%, KS 30–50%.

In the run below the score distribution barely moves (low PSI) while Gini and KS fall sharply. A stable score distribution does not prove the model still ranks well, so track Gini and KS on matured outcomes as well as PSI.

In [1]:
import numpy as np
from scipy import stats
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(0)
n = 20000; bad = rng.random(n) < 0.08
dev = np.where(bad, rng.normal(560, 45, n), rng.normal(620, 45, n))
later = np.where(bad, rng.normal(575, 50, n), rng.normal(612, 50, n))      # separation weakens and the population shifts
gini = lambda s: 2 * roc_auc_score(~bad, s) - 1
ks = lambda s: stats.ks_2samp(s[bad], s[~bad]).statistic
edges = np.quantile(dev, np.linspace(0, 1, 11)); edges[0], edges[-1] = -np.inf, np.inf
e, a = np.histogram(dev, edges)[0] / n, np.histogram(later, edges)[0] / n
print(f"Gini: development {gini(dev):.3f} -> later {gini(later):.3f} | KS: {ks(dev):.3f} -> {ks(later):.3f}")
print("PSI of the score distribution, development vs later:", round(float(np.sum((a - e) * np.log(a / e))), 3))

Gini: development 0.656 -> later 0.378 | KS: 0.502 -> 0.277
PSI of the score distribution, development vs later: 0.022


### Reject Inference: the Missing Outcomes Problem

A lender only observes repayment for applicants it approved. A model trained on approved applicants sees a restricted, safer population, and if the old approval decision depended on information the model does not have (an analyst's judgement, an old policy rule), its estimates for the rejected population are biased, not just noisy.

Synthetic example where an unobserved "judgement" signal raises default risk and also drove past approvals:

In [4]:
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
n = 80000
income = rng.lognormal(10.8, 0.5, n); util = rng.beta(2, 3, n); delinq = rng.poisson(0.4, n)
h = rng.normal(size=n)                                   # judgement signal: raises risk, not visible to the model
z = -1.8 - 1.5 * (np.log(income) - 10.8) + 2.0 * util + 0.5 * delinq + 1.2 * h
default = rng.binomial(1, 1 / (1 + np.exp(-z)))
X = pd.DataFrame({"log_income": np.log(income), "utilization": util, "delinquencies": delinq})
old_score = -(np.log(income) - 10.8) + util + 0.3 * delinq + 0.8 * h + rng.normal(scale=0.3, size=n)
approved = old_score < np.quantile(old_score, 0.5)       # past policy approved the safer half, using h too
rejected = ~approved

m_app = LogisticRegression(max_iter=1000).fit(X[approved], default[approved])
m_all = LogisticRegression(max_iter=1000).fit(X, default)    # oracle: needs outcomes for everyone
pa, pl = m_app.predict_proba(X)[:, 1], m_all.predict_proba(X)[:, 1]
print("observed default rate, approved:", default[approved].mean().round(3), "| true rate among rejected:", default[rejected].mean().round(3))
print("mean predicted PD over all applicants: approved-only model", pa.mean().round(3), "| oracle", pl.mean().round(3), "| actual", default.mean().round(3))
print("mean predicted PD on rejected: approved-only model", pa[rejected].mean().round(3), "| actual", default[rejected].mean().round(3))
print("AUC on rejected applicants: approved-only", round(roc_auc_score(default[rejected], pa[rejected]), 3), "| oracle", round(roc_auc_score(default[rejected], pl[rejected]), 3))

observed default rate, approved: 0.159 | true rate among rejected: 0.565
mean predicted PD over all applicants: approved-only model 0.184 | oracle 0.362 | actual 0.362
mean predicted PD on rejected: approved-only model 0.209 | actual 0.565
AUC on rejected applicants: approved-only 0.607 | oracle 0.606


The ranking (AUC) of the approved-only model is about as good as the oracle's, but the level is far off: it predicts less than half the true default rate for the rejected group. Anyone who prices or sets cutoffs from those PDs takes on far more risk than expected. Mitigations: reject inference methods (augmentation, parceling), using outcome data from other products or bureau performance on the rejected, and best of all a small randomized approval sample that gives unbiased outcomes.

### Fairness and Adverse Action

Two requirements shape any lending model:

- **Adverse action reasons.** When an applicant is declined, US rules require the principal reasons. A linear or scorecard model gives these directly (the variables that cost the applicant the most points). For complex models, teams use attribution methods such as SHAP.
- **Disparate impact.** A model with no protected attribute can still approve groups at very different rates. A common screening test is the four-fifths rule: the approval rate of a group divided by that of the highest group should not fall below 0.8. If it does, the lender must search for a less discriminatory alternative that keeps comparable predictive power.

Synthetic applicants with a group-correlated feature:

In [5]:
rng = np.random.default_rng(5)
n = 50000
group = rng.integers(0, 2, n)                                       # 0 = group A, 1 = group B (protected, not a model feature)
income = rng.lognormal(10.8 - 0.25 * group, 0.5)                    # group B has lower income on average
util = rng.beta(2, 3, n); delinq = rng.poisson(0.4, n)
z = -1.2 - 1.5 * (np.log(income) - 10.8) + 2.0 * util + 0.5 * delinq
default = rng.binomial(1, 1 / (1 + np.exp(-z)))
Xf = pd.DataFrame({"log_income": np.log(income), "utilization": util, "delinquencies": delinq})
model = LogisticRegression(max_iter=1000).fit(Xf, default)
pd_hat = model.predict_proba(Xf)[:, 1]
approve = pd_hat < np.quantile(pd_hat, 0.6)                          # approve the safest 60%

rate = pd.Series(approve).groupby(group).mean()
print("approval rate by group:", rate.round(3).to_dict())
air = rate.min() / rate.max()
print("adverse impact ratio:", round(air, 3), "->", "below 0.8, investigate" if air < 0.8 else "passes four-fifths screen")

mean = Xf.mean()
declined = np.where(~approve)[0][0]
contrib = pd.Series(model.coef_[0] * (Xf.iloc[declined] - mean), index=Xf.columns)   # contribution to risk vs the average applicant
reasons = contrib[contrib > 0].sort_values(ascending=False).head(2)               # only factors that pushed risk up
print("principal reasons for declining applicant", declined, ":", reasons.round(2).to_dict())

approval rate by group: {0: 0.678, 1: 0.523}
adverse impact ratio: 0.771 -> below 0.8, investigate
principal reasons for declining applicant 1 : {'log_income': 1.17}


The model never sees `group`, yet the income gap passes through and the approval rates differ. The ratio is a screen, not a verdict: the next step is to test whether another model or feature set keeps similar accuracy with a smaller gap.

### Blogs

- **Upstart.** Upstart says it selects model variables for their ability to predict default (it is reported to use non-traditional data such as education and employment history) and tests for adverse impact on protected classes and for less discriminatory alternatives. A fair-lending monitorship by Relman Colfax found approval disparities and recommended a less discriminatory model. Upstart's page on less discriminatory alternatives describes removing variables and tuning hyperparameters as approaches that did not give the needed result, and adversarial debiasing as the one it uses now: a main model predicts credit risk while a second model tries to detect protected attributes from the output. [Upstart: Less Discriminatory Alternatives](https://www.upstart.com/lenders/regulatory-compliance/less-discriminatory-alternatives), [Fair Lending Monitorship of Upstart Network's Lending Model](https://www.relmanlaw.com/media/news/1182_PUBLIC%20Upstart%20Monitorship_2nd%20Report_FINAL.pdf)
- **Zest AI.** Zest AI presents explainability as a compliance requirement for ML underwriting, stressing accurate adverse action reasons so that lenders can use more complex models while meeting fair-lending rules. [Zest AI: Model Explainability, Re-explained](https://www.zest.ai/learn/resources/model-explainability-reexplained/)

Common pattern: lenders move from scorecards toward gradient-boosted or neural models for accuracy, and pay for it with explanation tooling (reason codes, SHAP), fairness testing with a search for less discriminatory alternatives, and model-risk documentation.

#### Pitfalls

- Treating the approved population as the whole population (selection bias).
- Using a variable that proxies a protected attribute without testing its impact.
- A very high IV on one variable: check for leakage before celebrating.
- Judging the model by AUC alone when pricing needs calibrated PDs.
- Not monitoring the model after launch: population drift (PSI) and rank-ordering (Gini, KS) should be tracked, as in the scorecard validation questions.